In [1]:
import yaml

with open("../configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Usage examples
n_factors    = config["models"]["svd"]["n_factors"]
w_svd        = config["models"]["hybrid"]["w_svd"]
matrix_path  = config["artifacts"]["interaction_matrix"]

In [4]:
import pickle
import pandas as pd
import numpy as np
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

with open("../data/processed/hybrid_recommender.pkl", "rb") as f:
    hybrid_rec = pickle.load(f)

popularity_df = pd.read_parquet("../data/processed/artist_popularity.parquet")
content_df    = pd.read_parquet("../data/processed/content_features.parquet")
interactions  = pd.read_parquet("../data/processed/interactions.parquet")

sample_user = interactions["user_id"].iloc[100]

In [5]:
from src.recommenders.ranker import RecommendationRanker

ranker = RecommendationRanker(
    novelty_weight=0.2,
    diversity_lambda=0.7,
    dedup_threshold=0.85
)
ranker.fit(popularity_df, content_df)

In [6]:
import time

# Get larger candidate pool for ranker to work with
candidates = hybrid_rec.recommend(user_id=sample_user, k=50, candidate_pool=100)
print(f"Candidate pool size: {len(candidates)}")

start = time.time()
ranked = ranker.rank(candidates, k=10)
print(f"Ranking time: {time.time()-start:.3f}s")

print(f"\nFinal Ranked Recommendations:")
print(ranked.to_string(index=False))

Candidate pool size: 50
Ranking time: 0.035s

Final Ranked Recommendations:
              artist_name  ranked_score  hybrid_score  novelty_score
            jenifer lópez      0.635521      0.600000       0.777604
                  baladas      0.635521      0.600000       0.777604
                  shakira      0.613049      0.765742       0.002277
       julio iglesias jr.      0.589795      0.542843       0.777604
         marcos hernández      0.589795      0.542843       0.777604
                     Àëñó      0.589795      0.542843       0.777604
celine dion & bryan adams      0.589323      0.542252       0.777604
               fenyő iván      0.565042      0.511902       0.777604
            daniel zueras      0.564392      0.511089       0.777604
                   djelem      0.562088      0.508209       0.777604


In [7]:
hybrid_top10 = candidates.head(10)["artist_name"].tolist()
ranked_top10 = ranked["artist_name"].tolist()

print("BEFORE ranking (hybrid score order):")
for i, a in enumerate(hybrid_top10, 1):
    print(f"  {i:2}. {a}")

print("\nAFTER ranking (novelty + diversity):")
for i, a in enumerate(ranked_top10, 1):
    print(f"  {i:2}. {a}")

reorder_count = sum(1 for a in ranked_top10 if a not in hybrid_top10)
print(f"\nNew artists introduced by ranker: {reorder_count}/10")

BEFORE ranking (hybrid score order):
   1. shakira
   2. carlos baute
   3. baladas
   4. jenifer lópez
   5. fergie
   6. jose el frances
   7. beatriz luengo
   8. marcos hernández
   9. julio iglesias jr.
  10. Àëñó

AFTER ranking (novelty + diversity):
   1. jenifer lópez
   2. baladas
   3. shakira
   4. julio iglesias jr.
   5. marcos hernández
   6. Àëñó
   7. celine dion & bryan adams
   8. fenyő iván
   9. daniel zueras
  10. djelem

New artists introduced by ranker: 4/10


In [8]:
configs = [
    (False, False, False, "No ranking (hybrid order)"),
    (True,  False, False, "Dedup only"),
    (True,  True,  False, "Dedup + Novelty"),
    (True,  True,  True,  "Full ranking (Dedup + Novelty + Diversity)"),
]

for dedup, novelty, diversity, label in configs:
    result = ranker.rank(
        candidates, k=10,
        use_dedup=dedup,
        use_novelty=novelty,
        use_diversity=diversity
    )
    print(f"\n{label}:")
    print(result["artist_name"].tolist())


No ranking (hybrid order):
['shakira', 'carlos baute', 'baladas', 'jenifer lópez', 'fergie', 'jose el frances', 'beatriz luengo', 'marcos hernández', 'julio iglesias jr.', 'Àëñó']

Dedup only:
['shakira', 'carlos baute', 'baladas', 'jenifer lópez', 'fergie', 'jose el frances', 'beatriz luengo', 'marcos hernández', 'julio iglesias jr.', 'Àëñó']

Dedup + Novelty:
['jenifer lópez', 'baladas', 'shakira', 'carlos baute', 'julio iglesias jr.', 'marcos hernández', 'Àëñó', 'celine dion & bryan adams', 'fenyő iván', 'daniel zueras']

Full ranking (Dedup + Novelty + Diversity):
['jenifer lópez', 'baladas', 'shakira', 'julio iglesias jr.', 'marcos hernández', 'Àëñó', 'celine dion & bryan adams', 'fenyő iván', 'daniel zueras', 'djelem']


In [9]:
with open("../data/processed/ranker.pkl", "wb") as f:
    pickle.dump(ranker, f)
print("Ranker saved.")

Ranker saved.
